In [25]:
# Dependencies installed via: uv pip install openai python-dotenv requests minsearch

In [26]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv

In [27]:
load_dotenv()

openai_client = OpenAI()

In [28]:
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Tell me a short Christmas story"}
    ]
)

print(response.choices[0].message.content)

**The Christmas Star**

In a quaint village nestled between snowy mountains, the townsfolk eagerly prepared for Christmas. Every year, the highlight was the lighting of the giant Christmas tree in the town square. This year, however, things felt different. A heavy fog blanketed the village, hiding the stars and casting a shadow over the festivities.

Young Lucy, a bright-eyed girl with a heart full of hope, watched as her neighbors expressed their worries about the missing Christmas spirit. Determined to bring back the joy, she had an idea. “Let’s make our very own Christmas star!” she exclaimed.

With the help of her friends, Lucy gathered materials — twinkling lights, colorful paper, and a sturdy wooden frame. They worked tirelessly, laughing and sharing stories, all while the fog swirled around them. As the tree stood bare in the town square, the village felt gloomier than ever.

On Christmas Eve, after hours of hard work, Lucy and her friends unveiled their masterpiece: a large, ra

In [29]:
# Groq client (OpenAI-compatible API)
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

# Test Groq client
response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Tell me a short HannukaH story"}
    ]
)

print(response.choices[0].message.content)

Here's a classic Hanukkah story, one of the most well-known and beloved tales in Jewish tradition.

**The Miracle of the Oil**

The year was 164 BCE, and the Maccabean Revolt had just taken place in Jerusalem. The Syrian-Greek ruler, Antiochus IV, had desecrated the Temple in Jerusalem by erecting a statue of Zeus Olympios and forcing Jews to worship Greek gods.

A small group of brave Jewish rebels, led by Mattathias and his five sons, including Judah Maccabee, had risen up against their oppressors and miraculously defeated them in battle. The Temple was now in shambles, and the Jewish people were eager to rededicate it to their one true God.

When the Maccabees arrived at the Temple, they were met with a disturbing sight: the altar had been desecrated, and the sacred Temple menorah (candelabrum) had been smashed. But despite the destruction, they discovered a small bottle of pure olive oil that had been left untouched in a corner of the Temple. The oil had been consecrated for the Te

In [30]:
# Choose which client to use for the rest of the notebook
client = groq_client  # or groq_client

In [31]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

**The Unicorn and the Starry Night**

Once upon a time, in a magical forest where sunlight danced through the trees, there lived a gentle unicorn named Luna. Her coat sparkled like fresh snow, and her horn shimmered with all the colors of the rainbow.

One evening, as the sun began to dip below the horizon, casting a warm glow over the forest, Luna wandered to her favorite meadow. The sky transformed into a canvas of pinks and purples, and the first stars twinkled above. But as Luna frolicked beneath the vast sky, she noticed one little star in particular, flickering sadly.

“What’s wrong, twinkling star?” she called softly.

The star sighed, its tiny voice echoing like a whisper. “I feel lost. I’ve strayed too far from my constellation and can’t find my way home.”

Luna’s heart ached for the little star. “Don’t worry! I can help you get back.”

With a graceful leap, she spread her shimmering wings, which radiated a gentle light. “Climb on my back.”

The star sparkled with hope and swi

In [32]:
response = client.responses.create(
    model="openai/gpt-oss-20b",
    input="Write a short bedtime story about a unicorn."
)

print(response.output_text)

**The Moonlit Meadow and Luna the Unicorn**

Once upon a twilight, in a forest where the trees hummed lullabies, lived a gentle unicorn named Luna. Her coat was as silver as the moon, and her horn glowed like a tiny star. Every night, Luna would wander the moonlit meadow, her hooves whispering on dew‑soft grass.

One evening, as the sky painted itself in soft blues and purples, Luna heard a soft rustle. It was an old owl named Orion, perched on a silver birch. Orion was shy; he rarely left his tree because the forest’s noises made him nervous.

“Why do you stay in your branch?” Luna asked, her voice a soothing breeze.

“I’m afraid the dark is too big,” Orion whispered. “I wish I could see the stars.”

Luna smiled. “Let me show you.” She gently nudged her horn toward Orion, and a silver ribbon of light drifted from it, wrapping around the owl. Together, they floated above the forest, the stars twinkling like tiny lanterns.

They sang a soft song of moonlight and calm, and Orion’s feathe

In [ ]:
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()
print(documents_raw)

In [34]:
documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [35]:
documents[12]

{'text': 'The zoom link is only published to instructors/presenters/TAs.\nStudents participate via Youtube Live and submit questions to Slido (link would be pinned in the chat when Alexey goes Live). The video URL should be posted in the announcements channel on Telegram & Slack before it begins. Also, you will see it live on the DataTalksClub YouTube Channel.\nDon’t post your questions in chat as it would be off-screen before the instructors/moderators have a chance to answer it if the room is very active.',
 'section': 'General course-related questions',
 'question': 'Office Hours - What is the video/zoom link to the stream for the “Office Hour” or workshop sessions?',
 'course': 'data-engineering-zoomcamp'}

In [37]:
from minsearch import AppendableIndex

In [38]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [39]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
    )

    return results

In [40]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [42]:
question = 'I just discovered the course. Can I join it now?'

In [46]:
result = search(question)
print(result)

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.", 'section': 'General course-related questions', 'question': 'Course - Can I still join the course after the start date?', 'course': 'data-engineering-zoomcamp'}, {'text': "No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the course is running.", 'section': 'General course-related questions', 'question': 'Certificate - Can I follow the course in a self-paced mode and get a certificate?', 'course': 'data-engineering-zoomcamp'}, {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.

In [48]:
prompt = f"""
Answer the question from the student using the provided context

<QUESTION>{question}</QUESTION>

<CONTEXT>{json.dumps(result)}</CONTEXT>
"""

In [49]:
# agentic RAG

chat_messages = [
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

In [50]:
print(response)

Response(id='resp_0c779d099774a88300692584215e0081a0a954c490a9994269', created_at=1764066337.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"Can I join the course now?"}', call_id='call_sqm37OaWh498X2FyOsQ88eLp', name='search', type='function_call', id='fc_0c779d099774a8830069258422992081a0ba677c4ac83fe34f', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the course FAQ.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', description='Search the FAQ database')], top_p=1.0, background=False, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_reten

In [51]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course now?"}', call_id='call_sqm37OaWh498X2FyOsQ88eLp', name='search', type='function_call', id='fc_0c779d099774a8830069258422992081a0ba677c4ac83fe34f', status='completed')

In [52]:
chat_messages.append(tool_call)

In [53]:
search_result = search(query="Can I join the course now?")

In [54]:
result_json = json.dumps(search_result, indent=2)

chat_messages.append({
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": result_json,
})

In [55]:
chat_messages

[{'role': 'user',
  'content': 'I just discovered the course. Can I join it now?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course now?"}', call_id='call_sqm37OaWh498X2FyOsQ88eLp', name='search', type='function_call', id='fc_0c779d099774a8830069258422992081a0ba677c4ac83fe34f', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_sqm37OaWh498X2FyOsQ88eLp',
  'output': '[\n  {\n    "text": "Yes, even if you don\'t register, you\'re still eligible to submit the homeworks.\\nBe aware, however, that there will be deadlines for turning in the final projects. So don\'t leave everything for the last minute.",\n    "section": "General course-related questions",\n    "question": "Course - Can I still join the course after the start date?",\n    "course": "data-engineering-zoomcamp"\n  },\n  {\n    "text": "No, you can only get a certificate if you finish the course with a \\u201clive\\u201d cohort. We don\'t award certificates for the self-paced mode. T

In [56]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)

In [57]:
response.output_text

"Yes, you can still join the course! Even if you don't register, you're eligible to submit homework. Just keep in mind that there are deadlines for turning in the final projects, so it's a good idea not to leave everything until the last minute. \n\nIf you have any other questions, feel free to ask!"

In [58]:
chat_messages.append(
    {"role": "user", "content": "but are you sure I can get my certificate?"}
)

In [59]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=chat_messages,
    tools=[search_tool]
)
response.output_text

'To receive a certificate, you must complete the course with a "live" cohort. Certificates are not awarded for self-paced completion. You\'ll need to peer-review capstone projects during the course to qualify for the certificate.'